# Llama Sense Annotation Pipeline (Refactored)

This notebook uses the new modular pipeline for sense annotation with Llama, leveraging shared utilities for maintainability and traceability.

Use this notebook chunk by chunk for either sense repo round:
- Set `ROUND = 1` for the old/first-round sense repo or `ROUND = 2` for the current/second-round sense repo.
- Set `chunk_idx` to choose the corpus file: `0` for `0001-0500`, `1` for `0501-1000`, `2` for `1001-1500`, and so on.
- For a full chunk run, set `test = False`.
- For a partial rerun inside a chunk, set `test = True` and choose `tb` and `te` as 0-based offsets inside the selected chunk.
- The notebook converts those offsets back to absolute sentence numbers for filenames and logs. For example, `chunk_idx = 1`, `tb = 0`, `te = 100` writes outputs for `0501-0600`.
- For the full first chunk against the second-round sense repo, use `ROUND = 2`, `chunk_idx = 0`, `test = False`.

## Model Information (for Research Paper)

| Property | Value |
|----------|-------|
| **Model** | Meta Llama 4 Scout |
| **Ollama Tag** | `llama4:latest` |
| **Architecture** | llama4 (Mixture of Experts) |
| **Total Parameters** | 108.6B |
| **Active Parameters** | 17B |
| **Context Length** | 10,485,760 tokens |
| **Quantization** | Q4_K_M |
| **Temperature** | 0.0 |
| **Integration** | langchain-ollama 1.1.0 |
| **Release Date** | April 5, 2025 |
| **Knowledge Cutoff** | August 2024 |

In [1]:
from config import ANNOTATION_CHUNKS, DATA_DIR, OUTPUT_DIR
from data_loader import load_sense_repo_by_round
from process_senses import process_senses_with_chain, default_build_senses_block, parse_model_output
from writers import CustomWebAnnoTSVWriter, InceptionWebAnnoTSVWriter
import time

# Annotation round to use (1 = old/first round, 2 = current/second round)
ROUND = 2

# Set model origin for traceability
ORIGIN_LLM = "Llama4"

In [2]:
# Load sense repository and select corpus chunk
senses_df = load_sense_repo_by_round(round_number=ROUND)
print(f"Loaded sense repo for round {ROUND}: {len(senses_df)} senses")

from webanno_spacy_converter.parsers.tsv_parser_v3 import WebAnnoLEXISParser

chunks = [(b, e, (DATA_DIR / fname)) for (b, e, fname) in ANNOTATION_CHUNKS]
print("Available chunks (index, begin, end, file):",
      [(i, b, e, p.name) for i, (b, e, p) in enumerate(chunks)])

chunk_idx = 1  # 0 = 1-500, 1 = 501-1000, 2 = 1001-1500, ...
chunk_begin, chunk_end, selected_path = chunks[chunk_idx]
print(f"Using chunk #{chunk_idx}: {selected_path.name} -> ({chunk_begin}, {chunk_end})")

parser = WebAnnoLEXISParser(selected_path)
sentences = parser.parse()

begin, end = chunk_begin, chunk_end

# Run the first 100 sentences from the second chunk -> absolute range 501-600
test = True
if test:
    tb, te = 0, 100
    sentences = sentences[tb:te]
    begin = chunk_begin + tb
    end = chunk_begin + te - 1
    print(f"Using subset offsets {tb}:{te} -> absolute sentence range {begin}-{end}")
else:
    print(f"Using full chunk sentence range {begin}-{end}")

Loaded sense repo for round 2: 13283 senses
Available chunks (index, begin, end, file): [(0, 1, 500, 'sr-elexis-WSD_0001_0500.tsv'), (1, 501, 1000, 'sr-elexis-WSD_0501_1000.tsv'), (2, 1001, 1500, 'sr-elexis-WSD_1001_1500.tsv'), (3, 1501, 2000, 'sr-elexis-WSD_1501_2000.tsv'), (4, 2001, 2024, 'sr-elexis-WSD_2001_2024.tsv')]
Using chunk #1: sr-elexis-WSD_0501_1000.tsv -> (501, 1000)
Using subset offsets 0:100 -> absolute sentence range 501-600


In [3]:
from langchain_ollama import ChatOllama
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Compose Llama LLM chain using ChatPromptTemplate (let ChatOllama handle formatting)
system_message = """
Vi ste ekspert za leksiku i semantiku. Na osnovu konteksta rečenice i liste validnih značenja ciljne reči,
vaš zadatak je da identifikujete ono značenje koje se najpreciznije koristi u datom kontekstu.

Odgovor mora biti u strogo definisanom JSON formatu:

{{
  "sense_id": "<TAČAN ID iz liste ili 'NEW_SENSE'>",
  "explanation": "<kratko obrazloženje u jednoj ili dve rečenice>"
}}

VAŽNO:
- sense_id MORA biti IDENTIČAN jednom od ponuđenih ID-jeva (npr. "ENG30-00551215-n") ili tačno "NEW_SENSE"
- NIKADA ne koristite brojeve poput "1", "2", "značenje 1" itd.
- Ne dodajete nikakav tekst van JSON strukture
"""

user_prompt = """
Kontekst rečenice (ciljna reč je označena HTML tagom <b>...</b>):
"{sentence}"

Ciljna reč: "{word}"

Lista mogućih značenja:
{senses_block}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_message),
    ("user", user_prompt)
])

llm = ChatOllama(
    model="llama4",
    temperature=0.0,
)
parser = StrOutputParser()
chain = prompt | llm | parser

# Update origin label to reflect exact model
ORIGIN_LLM = "Llama4"

In [4]:
# Annotate senses using the shared utility
start_time = time.time()
sentences = process_senses_with_chain(
    sentences,
    senses_df,
    chain,
    ORIGIN_LLM,
    build_senses_block=default_build_senses_block,
    parse_json_response_clean=parse_model_output
)
end_time = time.time()
print(f"Processed sentences {begin} to {end} in {end_time - start_time:.2f} seconds.")

Processed 100/100 sentences.Processed sentences 501 to 600 in 17865.21 seconds.


In [5]:
# Save outputs for the selected round and chunk
ROUND_SUFFIX = f"_round{ROUND}"
writer = CustomWebAnnoTSVWriter(sentences)
writer.save(OUTPUT_DIR / f"LexiSense_{begin:04d}_{end:04d}_{ORIGIN_LLM}{ROUND_SUFFIX}.tsv")

In [6]:
# Write Inception-compatible output (for annotation import)
incept_writer = InceptionWebAnnoTSVWriter(sentences)
incept_writer.save(OUTPUT_DIR / f"LexiSense_Inception_{begin:04d}_{end:04d}_{ORIGIN_LLM}{ROUND_SUFFIX}.tsv")

## Log processing time and completion

In [7]:
with open(f"llama_round{ROUND}.log", "a", encoding="utf-8") as f:
    f.write(f"Chunk {chunk_idx} | processed sentences {begin} to {end}\n")
    f.write(f"Llama4 took {end_time - start_time:.2f} seconds\n")
    f.write(f"Or minutes: {(end_time - start_time)/60:.2f}\n")
print("Done.")

Done.
